# Data Engineering: ETL Pipeline

## Introduzione

Questo progetto ha l'obiettivo di sviluppare una pipeline ETL
(Extract, Transform, Load) utilizzando Python e MySQL.

La pipeline permette di acquisire dati da una fonte esterna,
effettuare operazioni di pulizia e trasformazione e caricare
i dati elaborati all'interno di un database relazionale.

## Obiettivi del progetto

- Acquisire dati da una fonte esterna.
- Analizzare e pulire i dati con Python e Pandas.
- Trasformare i dati per renderli coerenti e utilizzabili.
- Caricare i dati in un database MySQL.
- Gestire e verificare il processo ETL.
- Organizzare il codice secondo una struttura modulare.
- Documentare il progetto e renderlo disponibile su GitHub.

## Tecnologie utilizzate

- Python
- Pandas
- MySQL
- SQLAlchemy
- PyMySQL
- Jupyter Notebook
- Git e GitHub

## Architettura della pipeline

La pipeline seguirà tre fasi principali:

1. **Extract:** acquisizione dei dati dalla fonte originale.
2. **Transform:** pulizia e trasformazione dei dati.
3. **Load:** caricamento dei dati nel database MySQL.

## 1. Configurazione dell'ambiente e delle credenziali

Per proteggere le credenziali del database, utilizziamo
un file `.env` esterno al notebook.

Le variabili vengono caricate tramite la libreria `python-dotenv`,
evitando di inserire informazioni sensibili direttamente nel codice.

## 2. Importazione delle librerie

Importiamo le librerie necessarie per la gestione e la trasformazione
dei dati, la connessione al database MySQL e il monitoraggio
delle attività della pipeline ETL.

In [1]:
# Importazione delle librerie per la gestione dei dati

import os
import sys
import pandas as pd

# Importazione degli strumenti per la gestione dei percorsi

from pathlib import Path

# Importazione degli strumenti per la connessione al database

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Importazione degli strumenti per la registrazione delle attività

import logging

# Aggiungiamo la cartella src al percorso di ricerca dei moduli

sys.path.append(str(Path.cwd().parent / "src"))

# Importiamo il logger del progetto

from logger import logger

## 3. Caricamento delle variabili d'ambiente

Carichiamo le credenziali e i parametri di connessione
dal file `.env`, utilizzando la libreria `python-dotenv`.

Questa procedura consente di mantenere separate le informazioni
sensibili dal codice sorgente.

In [2]:
# Carichiamo le variabili d'ambiente dal file .env

load_dotenv()

# Recuperiamo i parametri di connessione al database

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

## 4. Configurazione della connessione al database MySQL

Utilizziamo SQLAlchemy per creare il motore di connessione
al database MySQL.

I parametri necessari vengono recuperati dalle variabili
d'ambiente caricate nella sezione precedente.

In [10]:
# Verifichiamo che tutte le variabili di connessione siano presenti

parametri_db = [
    DB_HOST,
    DB_PORT,
    DB_NAME,
    DB_USER,
    DB_PASSWORD
]

if not all(parametri_db):
    raise ValueError(
        "Configurazione incompleta: controllare il file .env"
    )

# Creiamo il motore di connessione a MySQL

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True
)

# Registriamo l'avvio della pipeline ETL

logger.info("Avvio della pipeline ETL")

# Registriamo la configurazione della connessione al database

logger.info("Connessione al database MySQL configurata")

2026-11-04 13:57:39,562 - INFO - Avvio della pipeline ETL
2026-11-04 13:57:39,563 - INFO - Connessione al database MySQL configurata


## 5. Verifica della connessione al database

Eseguiamo una query SQL per verificare che Python
riesca a comunicare correttamente con il database MySQL.

La verifica restituisce il nome del database attualmente selezionato.

In [11]:
# Verifichiamo la connessione al database MySQL

try:
    # Apriamo una connessione al database
    with engine.connect() as conn:

        # Recuperiamo il nome del database selezionato
        risultato = conn.execute(
            text("SELECT DATABASE()")
        )

        nome_database = risultato.scalar()

    # Confermiamo il risultato della verifica
    print("Connessione al database riuscita!")
    print(f"Database selezionato: {nome_database}")

except Exception as errore:
    # Interrompiamo il processo in caso di errore
    raise ConnectionError(
        "Impossibile stabilire la connessione al database MySQL."
    ) from errore

Connessione al database riuscita!
Database selezionato: data_engineering


## 6. Estrazione dei dati (Extract)

In questa fase acquisiamo i dati dalla sorgente originale
in formato CSV utilizzando la libreria Pandas.

Il dataset contiene informazioni relative agli ordini,
ai clienti, ai prodotti e alle transazioni commerciali.

I dati vengono caricati in un DataFrame senza modificare
la sorgente originale, preservando i dati grezzi
per le successive fasi della pipeline ETL.

In [12]:
# Definiamo il percorso principale del progetto

project_root = Path.cwd()

# Individuiamo la cartella principale del progetto

if not (project_root / "data").exists():
    project_root = project_root.parent

# Definiamo il percorso del dataset originale

file_path = project_root / "data" / "raw" / "superstore.csv"

# Verifichiamo che il file sorgente esista

if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset non trovato: {file_path}"
    )

# Estraiamo i dati dal file CSV

try:
    df_raw = pd.read_csv(
        file_path,
        encoding="utf-8-sig"
    )

except UnicodeDecodeError:
    df_raw = pd.read_csv(
        file_path,
        encoding="cp1252"
    )

# Registriamo il completamento dell'estrazione

logger.info(
    "Estrazione completata: %s record acquisiti",
    len(df_raw)
)

2026-11-04 13:57:47,212 - INFO - Estrazione completata: 9994 record acquisiti


### 6.1 Verifica dei dati estratti

Controlliamo le prime righe del dataset e le sue dimensioni
per verificare che l'acquisizione sia avvenuta correttamente.

In [13]:
# Visualizziamo un'anteprima dei dati estratti

display(df_raw.head())

# Verifichiamo le dimensioni del dataset

print(f"Numero di righe: {df_raw.shape[0]}")
print(f"Numero di colonne: {df_raw.shape[1]}")

,csvbase_row_id,﻿Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


Numero di righe: 9994
Numero di colonne: 22


## 7. Trasformazione dei dati (Transform)

In questa fase analizziamo e trasformiamo i dati estratti
per migliorarne la qualità e renderli adatti al caricamento
nel database MySQL.

Le operazioni previste comprendono:
- Analisi della struttura del dataset.
- Gestione dei valori mancanti.
- Rimozione dei record duplicati.
- Correzione dei tipi di dato.
- Verifica della coerenza dei dati.

    ### 7.1 Analisi preliminare della struttura dei dati

Esaminiamo le colonne, i tipi di dato e le informazioni
generali del dataset prima di procedere con le trasformazioni.

In [14]:
# Analizziamo la struttura generale del dataset

df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   csvbase_row_id  9994 non-null   int64  
 1   ﻿Row ID         9994 non-null   int64  
 2   Order ID        9994 non-null   object 
 3   Order Date      9994 non-null   object 
 4   Ship Date       9994 non-null   object 
 5   Ship Mode       9994 non-null   object 
 6   Customer ID     9994 non-null   object 
 7   Customer Name   9994 non-null   object 
 8   Segment         9994 non-null   object 
 9   Country         9994 non-null   object 
 10  City            9994 non-null   object 
 11  State           9994 non-null   object 
 12  Postal Code     9994 non-null   int64  
 13  Region          9994 non-null   object 
 14  Product ID      9994 non-null   object 
 15  Category        9994 non-null   object 
 16  Sub-Category    9994 non-null   object 
 17  Product Name    9994 non-null   o

### 7.2 Analisi dei valori mancanti

Identifichiamo la presenza di valori mancanti nelle colonne
per valutare eventuali interventi di pulizia.

In [15]:
# Calcoliamo il numero di valori mancanti per ogni colonna

valori_mancanti = df_raw.isnull().sum()

# Calcoliamo la percentuale di valori mancanti

percentuale_mancanti = (
    df_raw.isnull().mean() * 100
).round(2)

# Creiamo un riepilogo dei valori mancanti

riepilogo_mancanti = pd.DataFrame({
    "Valori mancanti": valori_mancanti,
    "Percentuale (%)": percentuale_mancanti
})

display(riepilogo_mancanti)

,Valori mancanti,Percentuale (%)
csvbase_row_id,0,0.0
﻿Row ID,0,0.0
Order ID,0,0.0
Order Date,0,0.0
Ship Date,0,0.0
Ship Mode,0,0.0
Customer ID,0,0.0
Customer Name,0,0.0
Segment,0,0.0
Country,0,0.0


### 7.3 Analisi dei record duplicati

Verifichiamo la presenza di record duplicati
per individuare eventuali anomalie nel dataset originale.

In [16]:
# Calcoliamo il numero di record duplicati

numero_duplicati = df_raw.duplicated().sum()

# Visualizziamo il risultato

print(f"Record duplicati individuati: {numero_duplicati}")

Record duplicati individuati: 0


### 7.4 Pulizia e standardizzazione delle colonne

In questa fase eliminiamo la colonna tecnica aggiuntiva
`csvbase_row_id`, mantenendo l'identificativo originale `Row ID`.

Standardizziamo inoltre i nomi delle colonne per facilitarne
la gestione nelle successive operazioni di trasformazione
e caricamento nel database.

In [17]:
# Creiamo una copia del dataset originale per le trasformazioni

df_clean = df_raw.copy()

# Rimuoviamo la colonna tecnica aggiuntiva

df_clean = df_clean.drop(columns=["csvbase_row_id"])

# Rimuoviamo eventuali caratteri invisibili dai nomi delle colonne

df_clean.columns = df_clean.columns.str.replace("\ufeff", "", regex=False)

# Standardizziamo i nomi delle colonne

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Verifichiamo i nomi delle colonne trasformate

df_clean.columns.tolist()

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub-category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit']

### 7.5 Conversione e standardizzazione dei tipi di dato

Convertiamo le colonne temporali nel formato datetime
e il codice postale in formato testuale.

Queste operazioni garantiscono una gestione coerente
dei dati e facilitano il successivo caricamento in MySQL.

In [18]:
# Standardizziamo il nome della colonna sub-category

df_clean = df_clean.rename(
    columns={"sub-category": "sub_category"}
)

# Convertiamo le colonne temporali in formato datetime

df_clean["order_date"] = pd.to_datetime(
    df_clean["order_date"],
    errors="raise"
)

df_clean["ship_date"] = pd.to_datetime(
    df_clean["ship_date"],
    errors="raise"
)

# Convertiamo il codice postale in formato testuale

df_clean["postal_code"] = (
    df_clean["postal_code"].astype("string")
)

### 7.6 Verifica della coerenza dei dati

Controlliamo il rispetto delle principali regole di qualità
relative a quantità, vendite, sconti e date degli ordini.

Queste verifiche permettono di individuare eventuali
incongruenze prima del caricamento nel database.

In [19]:
# Verifichiamo la coerenza dei principali campi numerici

controlli_qualita = {
    "Quantità non positive": (df_clean["quantity"] <= 0).sum(),
    "Vendite negative": (df_clean["sales"] < 0).sum(),
    "Sconti fuori intervallo": (
        (df_clean["discount"] < 0) |
        (df_clean["discount"] > 1)
    ).sum(),
    "Date di spedizione precedenti all'ordine": (
        df_clean["ship_date"] < df_clean["order_date"]
    ).sum()
}

# Creiamo un riepilogo dei controlli

riepilogo_qualita = pd.DataFrame(
    list(controlli_qualita.items()),
    columns=["Controllo", "Anomalie individuate"]
)

# Visualizziamo i risultati

display(riepilogo_qualita)

# Registriamo il completamento della trasformazione

logger.info(
    "Trasformazione completata: %s record elaborati",
    len(df_clean)
)

,Controllo,Anomalie individuate
0,Quantità non positive,0
1,Vendite negative,0
2,Sconti fuori intervallo,0
3,Date di spedizione precedenti all'ordine,0


2026-11-04 14:01:23,186 - INFO - Trasformazione completata: 9994 record elaborati


## 8. Preparazione dei dati per il caricamento (Load)

In questa fase salviamo il dataset trasformato in formato CSV
all'interno della cartella `data/processed`.

Il file rappresenta il risultato delle operazioni di pulizia
e standardizzazione effettuate durante la fase Transform.

La sorgente originale rimane invariata.

In [25]:
# Definiamo il percorso della cartella dei dati elaborati

processed_dir = project_root / "data" / "processed"

# Creiamo la cartella se non esiste già

processed_dir.mkdir(parents=True, exist_ok=True)

# Definiamo il percorso del file CSV elaborato

processed_file = processed_dir / "superstore_clean.csv"

# Salviamo il DataFrame trasformato in formato CSV

df_clean.to_csv(
    processed_file,
    index=False,
    encoding="utf-8"
)

# Registriamo il completamento del salvataggio

logger.info(
    "Salvataggio CSV completato: %s record salvati in %s",
    len(df_clean),
    processed_file
)

2026-11-04 14:07:06,755 - INFO - Salvataggio CSV completato: 9994 record salvati in C:\Users\MSI\Desktop\Progetti\data-engineering\data\processed\superstore_clean.csv


## 9. Caricamento dei dati nel database MySQL (Load)

In questa fase carichiamo il dataset trasformato
all'interno della tabella `superstore_sales`.

Utilizziamo Pandas e SQLAlchemy per trasferire i dati
dal file CSV al database MySQL.

La procedura prevede il caricamento dei record
e la gestione delle eventuali anomalie durante l'operazione.

In [26]:
# Definiamo il nome della tabella di destinazione

table_name = "superstore_sales"

# Registriamo l'avvio del caricamento

logger.info("Avvio del caricamento dei dati su MySQL")

try:
    # Avviamo una transazione sul database

    with engine.begin() as conn:

        # Svuotiamo la tabella prima del nuovo caricamento

        conn.execute(
            text(f"DELETE FROM {table_name}")
        )

        # Carichiamo il dataset trasformato nel database

        df_clean.to_sql(
            name=table_name,
            con=conn,
            if_exists="append",
            index=False,
            chunksize=1000,
            method="multi"
        )

    # Registriamo il completamento del caricamento

    logger.info(
        "Caricamento su MySQL completato: %s record caricati",
        len(df_clean)
    )

except Exception as errore:

    # Registriamo l'eventuale errore

    logger.exception(
        "Errore durante il caricamento dei dati su MySQL"
    )

    raise

2026-11-04 14:07:18,529 - INFO - Avvio del caricamento dei dati su MySQL
2026-11-04 14:07:22,094 - INFO - Caricamento su MySQL completato: 9994 record caricati


## 10. Verifica dell'integrità dei dati

In questa fase confrontiamo il numero di record presenti
nel DataFrame trasformato con quelli caricati nel database MySQL.

L'obiettivo è verificare che il processo di caricamento
abbia trasferito correttamente tutti i record.

In [27]:
# Recuperiamo il numero di record presenti nel DataFrame

record_python = len(df_clean)

# Recuperiamo il numero di record presenti nella tabella MySQL

with engine.connect() as conn:
    risultato = conn.execute(
        text("SELECT COUNT(*) FROM superstore_sales")
    )

    record_mysql = risultato.scalar()

# Confrontiamo i risultati

verifica_caricamento = record_python == record_mysql

# Visualizziamo il riepilogo

print(f"Record nel DataFrame Python: {record_python}")
print(f"Record nel database MySQL: {record_mysql}")
print(f"Caricamento verificato: {verifica_caricamento}")

Record nel DataFrame Python: 9994
Record nel database MySQL: 9994
Caricamento verificato: True


### 10.1 Verifica dell'unicità della chiave primaria

Verifichiamo che ogni record caricato nel database
abbia un identificativo `row_id` univoco.

La chiave primaria garantisce l'identificazione
univoca di ogni record nella tabella.

In [28]:
# Verifichiamo il numero totale di record e gli identificativi distinti

with engine.connect() as conn:

    risultato = conn.execute(
        text("""
            SELECT
                COUNT(*) AS totale_record,
                COUNT(DISTINCT row_id) AS identificativi_univoci
            FROM superstore_sales
        """)
    )

    totale_record, identificativi_univoci = risultato.fetchone()

# Verifichiamo l'unicità degli identificativi

chiave_primaria_valida = (
    totale_record == identificativi_univoci
)

# Visualizziamo il risultato

print(f"Record totali: {totale_record}")
print(f"Identificativi univoci: {identificativi_univoci}")
print(f"Chiave primaria verificata: {chiave_primaria_valida}")

Record totali: 9994
Identificativi univoci: 9994
Chiave primaria verificata: True


In [29]:
# Registriamo il completamento dell'intera pipeline ETL

logger.info("Pipeline ETL completata con successo")

2026-11-04 14:07:58,548 - INFO - Pipeline ETL completata con successo
